In [ ]:
"""
LLM Integration Module
Handles OpenAI API with Cohere fallback
"""

import os
from dotenv import load_dotenv
from openai import OpenAI
import cohere

load_dotenv()


class LLMIntegration:
    """Manages LLM API calls with fallback support"""

    def __init__(self):
        self.openai_client = None
        self.cohere_client = None
        self._initialize_clients()

    def _initialize_clients(self):
        """Initialize API clients"""
        openai_key = os.getenv("OPENAI_API_KEY")
        cohere_key = os.getenv("COHERE_API_KEY")

        if openai_key:
            self.openai_client = OpenAI(api_key=openai_key)
            print("OpenAI client initialized")
        else:
            print("Warning: No OpenAI API key found")

        if cohere_key:
            self.cohere_client = cohere.Client(cohere_key)
            print("Cohere client initialized")
        else:
            print("Warning: No Cohere API key found")

    def generate(self, prompt, model="gpt-4o-mini", temperature=0.8):
        """Generate content with OpenAI, fallback to Cohere"""
        # Try OpenAI first
        if self.openai_client:
            try:
                response = self.openai_client.chat.completions.create(
                    model=model,
                    messages=[
                        {
                            "role": "system",
                            "content": "You are a skilled brand content strategist. You write authentic, human-sounding content that avoids generic AI patterns."
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    temperature=temperature
                )
                return {
                    "provider": "OpenAI",
                    "model": model,
                    "content": response.choices[0].message.content
                }
            except Exception as e:
                print(f"OpenAI failed: {e}")
                print("Switching to Cohere fallback...")

        # Fallback to Cohere
        if self.cohere_client:
            try:
                response = self.cohere_client.generate(
                    model="command",
                    prompt=prompt,
                    max_tokens=1000,
                    temperature=temperature
                )
                return {
                    "provider": "Cohere",
                    "model": "command",
                    "content": response.generations[0].text.strip()
                }
            except Exception as e:
                print(f"Cohere also failed: {e}")

        return {
            "provider": "None",
            "model": "None",
            "content": "Error: Both OpenAI and Cohere failed to generate content."
        }

    def generate_with_review(self, prompt, model="gpt-4o-mini"):
        """Generate content and present for human review"""
        result = self.generate(prompt, model)

        print(f"\n{'='*60}")
        print(f"Provider: {result['provider']} | Model: {result['model']}")
        print(f"{'='*60}")
        print(result["content"])
        print(f"{'='*60}")

        # Human in the loop
        print("\nOptions:")
        print("  [a] Accept this content")
        print("  [r] Regenerate")
        print("  [e] Edit prompt and regenerate")

        choice = input("\nYour choice (a/r/e): ").strip().lower()

        if choice == "r":
            print("Regenerating...")
            return self.generate_with_review(prompt, model)
        elif choice == "e":
            new_input = input("Add feedback or instructions: ")
            updated_prompt = prompt + f"\n\nAdditional instructions: {new_input}"
            return self.generate_with_review(updated_prompt, model)
        else:
            return result